# Experiment 4 - Part 3: Evaluating RAG with RAGAs

**Course:** HAAI302COM - Artificial Intelligence Laboratory

"It looks right" is not an evaluation. A RAG system has two components that can fail separately, the **retriever** and the **generator**, and a single accuracy number hides which one is at fault. **RAGAs** (Retrieval-Augmented Generation Assessment) scores each side using an LLM as the judge, so no human has to label every answer.

| Metric | Question it answers | Needs | Component |
|---|---|---|---|
| **Faithfulness** | Is every claim in the answer supported by the retrieved context? | question, answer, contexts | Generator |
| **Answer relevancy** | Does the answer actually address the question? | question, answer | Generator |
| **Context precision** | Are the relevant chunks ranked at the top of what we retrieved? | question, reference, contexts | Retriever |
| **Context recall** | Does the retrieved context contain everything the reference answer needs? | question, reference, contexts | Retriever |

All four return a number between 0 and 1 (higher is better). In this notebook we first open each metric up on small examples so you can see what the judge LLM is asked, and then we use them to compare four RAG configurations on 12 questions.

## 0. Setup

**A compatibility note.** `ragas` 0.4.x still imports a Vertex AI class that recent `langchain-community` releases removed. We never use Vertex AI, so the cell below registers an empty stand-in module *before* importing `ragas`. If your versions do not have the problem, the `try` simply succeeds and nothing is replaced.

In [1]:
# !pip install -q -U ragas langchain-chroma langchain-huggingface langchain-openai chromadb pypdf sentence-transformers diskcache
import os, sys, types, json, re, asyncio, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 170)

import langchain_community.llms as _llms
try:
    from langchain_community.chat_models.vertexai import ChatVertexAI      # noqa: F401
except ImportError:
    _stub = types.ModuleType("langchain_community.chat_models.vertexai")
    _stub.ChatVertexAI = type("ChatVertexAI", (), {})
    sys.modules[_stub.__name__] = _stub
    _llms.VertexAI = type("VertexAI", (), {})
import ragas
print("ragas", ragas.__version__)

ragas 0.4.3


### The judge LLM and the embedding model

RAGAs needs an LLM to act as the judge. We pass it a Gemini client through the same OpenAI-compatible endpoint we used for generation; `llm_factory` wraps any OpenAI-style client into the object RAGAs expects (it adds structured-output parsing, so the judge's verdicts come back as typed fields instead of free text). Answer relevancy also needs an embedding model, and we reuse the local MiniLM.

The judge's replies are cached on disk in `ragas_cache/`, so re-running costs nothing and gives identical scores.

In [2]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.cache import DiskCacheBackend
from ragas.embeddings import HuggingFaceEmbeddings as RagasHFEmbeddings
from llm_setup import GEMINI_URL, get_llm, enable_cache, RAG_PROMPT, format_docs
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))

client = AsyncOpenAI(api_key=os.environ["GEMINI_API_KEY"], base_url=GEMINI_URL, timeout=90, max_retries=5)
judge = llm_factory("gemini-2.5-flash-lite", client=client, cache=DiskCacheBackend("ragas_cache"))
ragas_emb = RagasHFEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")
print(type(judge).__name__, "|", type(ragas_emb).__name__)

InstructorLLM | HuggingFaceEmbeddings


In [3]:
from ragas.metrics.collections import (Faithfulness, AnswerRelevancy,
                                       ContextPrecisionWithReference, ContextRecall)

faith = Faithfulness(llm=judge)
relev = AnswerRelevancy(llm=judge, embeddings=ragas_emb)       # strictness=3 questions generated per answer
prec = ContextPrecisionWithReference(llm=judge)
recall = ContextRecall(llm=judge)
print([m.name for m in (faith, relev, prec, recall)])

['faithfulness', 'answer_relevancy', 'context_precision_with_reference', 'context_recall']


Now load the vector store from Part 1, the question set, and the RAG chain pieces from Part 2.

In [4]:
from operator import itemgetter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

cache = enable_cache("llm_cache.json")                          # generator responses are cached too
llm = get_llm("gemini-2.5-flash-lite", temperature=0.0)

minilm = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
store_minilm = Chroma(collection_name="rag_papers", embedding_function=minilm, persist_directory="chroma_db")
qa = json.load(open("qa_set.json"))["answerable"]
print(store_minilm._collection.count(), "chunks |", len(qa), "questions")

237 chunks | 12 questions


## 1. Faithfulness: opening the black box

Faithfulness is a two-step procedure, and both steps are LLM calls:

1. **Statements:** the judge breaks the answer into short, self-contained factual statements.
2. **Verdicts:** for each statement the judge decides whether it can be inferred from the retrieved context (1) or not (0).

`faithfulness = supported statements / all statements`

We use a real question from the set and the chunks our retriever returns for it. The two underscore methods below are internal to RAGAs; we call them directly only so that we can look inside.

In [5]:
q = qa[7]["question"]                                            # which matrices does LoRA adapt?
ctx_docs = store_minilm.as_retriever(search_kwargs={"k": 4}).invoke(q)
contexts = [d.page_content for d in ctx_docs]
good = "LoRA adapts the query and value projection matrices, Wq and Wv, in most of its experiments."
print(q, "\nretrieved chunks:", len(contexts), "| total characters:", sum(map(len, contexts)))

Which weight matrices does LoRA adapt in most of its experiments? 
retrieved chunks: 4 | total characters: 2810


In [6]:
async def faith_detail(answer):
    # step 1: answer -> statements; step 2: each statement judged against the joined context
    statements = await faith._create_statements(q, answer)
    verdicts = await faith._create_verdicts(statements, "\n".join(contexts))
    return pd.DataFrame([{"statement": v.statement, "verdict": v.verdict, "reason": v.reason[:90]} for v in verdicts.statements])

df_good = await faith_detail(good)
df_good

,statement,verdict,reason
0,"LoRA adapts the query projection matrix, Wq, in most experiments.",1,"The context states: ""As mentioned in Section 4.2, we only apply LoRA toWq an..."
1,"LoRA adapts the value projection matrix, Wv, in most experiments.",1,"The context states: ""As mentioned in Section 4.2, we only apply LoRA toWq an..."


**What to notice:** the judge split our one-sentence answer into two atomic statements (one about Wq, one about Wv) and found support for both in the retrieved text, so faithfulness is 2 out of 2 = 1.0. Read the `reason` column: it quotes the sentence from the context that justifies each verdict. That quote is how you audit the judge.

Now the same answer with a made-up sentence added at the end. The extra claims are plausible but appear nowhere in the context.

In [7]:
halluc = good + " It also adapts the feed-forward layers and was first proposed by Google in 2019."
df_h = await faith_detail(halluc)
print("faithfulness =", round(df_h.verdict.mean(), 2), "(supported statements / all statements)")
df_h

faithfulness = 0.5 (supported statements / all statements)


,statement,verdict,reason
0,"LoRA adapts the query projection matrix, Wq, in most experiments.",1,"The context states: ""As mentioned in Section 4.2, we only apply LoRA toWq an..."
1,"LoRA adapts the value projection matrix, Wv, in most experiments.",1,"The context states: ""As mentioned in Section 4.2, we only apply LoRA toWq an..."
2,LoRA adapts feed-forward layers.,0,The context mentions applying LoRA to weight matrices in the self-attention ...
3,LoRA was first proposed by Google in 2019.,0,The provided text does not contain any information about when or by whom LoR...


**What to notice:** four statements this time. The first two are supported. "Adapts feed-forward layers" is not, because the context only lists the attention matrices, and "first proposed by Google in 2019" appears nowhere in the text. 2 supported out of 4 gives 0.5. Two points to remember. First, faithfulness is measured against the **context**, not against the real world: a statement that happens to be true but is absent from the retrieved text still scores 0. Second, this metric judges only the generator; it cannot tell us whether the retriever fetched the right chunks.

## 2. Answer relevancy: does the answer address the question?

This metric does **not** look at the context or the truth. The judge reads the answer and writes the questions that this answer would be a good reply to (3 of them, `strictness=3`). Each generated question is embedded and compared by cosine similarity with the original question; the score is the mean similarity. Answers that dodge the question get a score of 0 through a separate "noncommittal" check.

We try four answers to the same question.

In [8]:
answers = {
    "direct, correct": good,
    "correct but padded": good + " Self-attention lets every token look at every other token, and this is why Transformers replaced RNNs for most language tasks.",
    "off topic": "Transformers process sequences in parallel using self-attention, which makes them faster to train than recurrent networks.",
    "refusal": "I don't know based on the provided documents.",
}
rel = {name: (await relev.ascore(user_input=q, response=a)).value for name, a in answers.items()}
pd.Series(rel, name="answer_relevancy").round(3).to_frame()

,answer_relevancy
"direct, correct",0.909
correct but padded,0.646
off topic,0.087
refusal,0.000


**What to notice:** the direct answer scores 0.91 and the off-topic answer 0.09: the questions regenerated from the off-topic sentence are about Transformers in general, far from ours. The refusal scores exactly 0 because RAGAs treats a noncommittal answer as not relevant. Padding the answer with a true but unrelated sentence lowers the score from 0.91 to 0.65, because the regenerated questions now partly ask about self-attention and RNNs. So this metric rewards **focused** answers. It does not check correctness: a fluent, on-topic, wrong answer would score high.

## 3. Context precision: is the good stuff ranked first?

For each retrieved chunk, in rank order, the judge decides whether it is useful for answering the question against the reference (1 or 0). The score is the **average precision**: for every relevant chunk, take the fraction of relevant chunks among the ones up to its rank, and average these. It rewards putting relevant chunks at the top, because a retriever that buries the answer at rank 4 makes the prompt longer and noisier.

First the arithmetic alone, with hand-written verdict lists (no LLM involved):

In [9]:
cases = {"[1, 0, 0, 0]  relevant chunk first": [1, 0, 0, 0],
         "[0, 0, 0, 1]  relevant chunk last": [0, 0, 0, 1],
         "[1, 1, 0, 0]  two relevant, on top": [1, 1, 0, 0],
         "[1, 0, 1, 0]  relevant at ranks 1 and 3": [1, 0, 1, 0],
         "[0, 0, 0, 0]  nothing relevant": [0, 0, 0, 0]}
ap = {name: round(prec._calculate_average_precision(v), 3) for name, v in cases.items()}
pd.Series(ap, name="context_precision").to_frame()

,context_precision
"[1, 0, 0, 0] relevant chunk first",1.000
"[0, 0, 0, 1] relevant chunk last",0.250
"[1, 1, 0, 0] two relevant, on top",1.000
"[1, 0, 1, 0] relevant at ranks 1 and 3",0.833
"[0, 0, 0, 0] nothing relevant",0.000


**What to notice:** position matters. A single relevant chunk gives 1.0 at rank 1 but only 0.25 at rank 4. `[1, 1, 0, 0]` also gives 1.0 because every relevant chunk comes before every irrelevant one. `[1, 0, 1, 0]` gives 0.83: at rank 1 the share of relevant chunks so far is 1/1, at rank 3 it is 2/3, and the average of 1 and 0.67 is 0.83. This is the average-precision formula, written in words.

Now with the real judge. We take the retrieved chunks for our question, and compare their natural order with the order reversed and with an added irrelevant chunk from the other paper placed first.

In [10]:
ref = qa[7]["reference"]
noise = store_minilm.as_retriever(search_kwargs={"k": 1, "filter": {"source": "data/rag_lewis_2020.pdf"}}).invoke(q)[0].page_content
variants = {"retriever order": contexts, "reversed order": contexts[::-1], "noise chunk first": [noise] + contexts}
prec_scores = {name: (await prec.ascore(user_input=q, reference=ref, retrieved_contexts=c)).value for name, c in variants.items()}
pd.Series(prec_scores, name="context_precision").round(3).to_frame()

,context_precision
retriever order,0.833
reversed order,0.500
noise chunk first,0.500


**What to notice:** the retriever's own order scores 0.83 (relevant chunks at ranks 1 and 3). Reversing the very same four chunks drops the score to 0.50, and putting one irrelevant chunk in front gives 0.50 as well. In the first two rows the set of chunks is identical; only the order changed. This is why production systems often add a **reranker** after the retriever.

## 4. Context recall: did we retrieve everything we need?

The judge splits the **reference answer** into sentences and checks, for each one, whether it can be attributed to the retrieved context. `recall = attributable sentences / all reference sentences`. A reference sentence that appears nowhere in the context means the retriever missed some evidence, and no generator could have answered completely.

We compare: the normal retrieved context, only the first chunk (k=1), and chunks taken from the wrong paper.

In [11]:
wrong = [d.page_content for d in store_minilm.as_retriever(search_kwargs={"k": 4, "filter": {"source": "data/rag_lewis_2020.pdf"}}).invoke(q)]
ctx_variants = {"k=4 (retriever)": contexts, "k=1 (top chunk only)": contexts[:1], "wrong paper": wrong}
rec = {name: (await recall.ascore(user_input=q, reference=ref, retrieved_contexts=c)).value for name, c in ctx_variants.items()}
print("reference:", ref)
pd.Series(rec, name="context_recall").round(3).to_frame()

reference: In most experiments LoRA is applied only to the query and value projection matrices, Wq and Wv, of the self-attention module.


,context_recall
k=4 (retriever),1.0
k=1 (top chunk only),1.0
wrong paper,0.0


**What to notice:** the normal context and the top chunk alone both score 1.0: the single best chunk already contains everything the reference says, so retrieving four added no recall (only noise, which context precision above penalised). Chunks from the wrong paper score 0.0 because nothing in them supports the reference. Recall can only be computed with a reference answer, which is why we wrote one for every question.

## 5. Four RAG configurations to compare

Time to use the metrics for a decision. Notebook 01 found that the small MiniLM embedder misses the right chunk for many questions. So we compare it against Gemini's embedding model (`gemini-embedding-001`), and we vary `k` for the better one.

| Config | Embedder | k |
|---|---|---|
| `minilm_k4` | MiniLM (local, 384-dim) | 4 |
| `gemini_k2` | gemini-embedding-001 (API, 3072-dim) | 2 |
| `gemini_k4` | gemini-embedding-001 | 4 |
| `gemini_k8` | gemini-embedding-001 | 8 |

Everything else (chunks, prompt, generator model, temperature 0) is identical, so any difference in scores comes from retrieval. We copy the 237 chunks from the existing store and embed them with Gemini, saving the result to `chroma_db_gemini/` for re-use.

In [12]:
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

gem_emb = OpenAIEmbeddings(model="gemini-embedding-001", api_key=os.environ["GEMINI_API_KEY"], base_url=GEMINI_URL,
                           check_embedding_ctx_length=False, chunk_size=50)

raw = store_minilm._collection.get(include=["documents", "metadatas"])
chunks = [Document(page_content=t, metadata=m) for t, m in zip(raw["documents"], raw["metadatas"])]

if os.path.isdir("chroma_db_gemini"):
    store_gemini = Chroma(collection_name="rag_papers_gemini", embedding_function=gem_emb, persist_directory="chroma_db_gemini")
else:
    store_gemini = Chroma.from_documents(chunks, gem_emb, collection_name="rag_papers_gemini", persist_directory="chroma_db_gemini",
                                         collection_metadata={"hnsw:space": "cosine"})
print("gemini store:", store_gemini._collection.count(), "chunks")

gemini store: 237 chunks


Before spending any LLM calls, a quick retrieval-only check: the keyword hit rate from Notebook 01, now for both embedders at several `k`.

In [13]:
norm = lambda t: re.sub(r"\s+", " ", t).lower()
rows = []
for name, st in [("minilm", store_minilm), ("gemini", store_gemini)]:
    row = {"embedder": name}
    for k in [1, 2, 4, 8]:
        found = [any(norm(it["keyword"]) in norm(d.page_content) for d in st.similarity_search(it["question"], k=k)) for it in qa]
        row[f"hit@{k}"] = round(sum(found) / len(qa), 2)
    rows.append(row)
pd.DataFrame(rows)

,embedder,hit@1,hit@2,hit@4,hit@8
0,minilm,0.08,0.17,0.58,0.67
1,gemini,0.42,0.67,0.83,0.83


**What to notice:** Gemini's embedding model finds the keyword in the top chunk for 42% of the questions against 8% for MiniLM, and within four chunks for 83% against 58%. Beyond four chunks neither improves (0.83 and 0.67 at k=8), so the remaining misses are not solved by fetching more: those chunks are simply not close to the question in embedding space. This is a cheap keyword check; the RAGAs metrics below confirm it with an LLM judge.

Now the generation chain, identical to Part 2, wrapped in a function so we can build one per configuration.

In [14]:
def make_rag(store, k):
    retriever = store.as_retriever(search_kwargs={"k": k})
    return (RunnablePassthrough.assign(docs=itemgetter("question") | retriever)
            .assign(answer=RunnableLambda(lambda x: {"context": format_docs(x["docs"]), "question": x["question"]})
                           | RAG_PROMPT | llm | StrOutputParser()))

configs = {"minilm_k4": (store_minilm, 4), "gemini_k2": (store_gemini, 2),
           "gemini_k4": (store_gemini, 4), "gemini_k8": (store_gemini, 8)}
inputs = [{"question": it["question"]} for it in qa]

runs = []
for name, (st, k) in configs.items():
    outs = make_rag(st, k).batch(inputs, config={"max_concurrency": 4})
    for i, (it, o) in enumerate(zip(qa, outs)):
        runs.append({"config": name, "qid": i, "question": it["question"], "reference": it["reference"],
                     "answer": o["answer"], "contexts": [d.page_content for d in o["docs"]]})
runs = pd.DataFrame(runs)
print(runs.shape, "| refusals:", runs.groupby("config").answer.apply(lambda s: s.str.contains("I don't know").sum()).to_dict())

(48, 6) | refusals: {'gemini_k2': 0, 'gemini_k4': 0, 'gemini_k8': 0, 'minilm_k4': 2}


**What to notice:** 48 answers, 4 configurations times 12 questions. Only the MiniLM configuration answers "I don't know" (twice). The Gemini configurations never refuse. That is not automatically good: an answer that never refuses either has the evidence or is making something up, and faithfulness will tell us which.

## 6. Scoring all answers with RAGAs

We score every row with all four metrics. The judge calls are `async`, so we launch them concurrently with a limit of 6 at a time (`asyncio.Semaphore`) to stay inside the API rate limit. A failed call becomes `NaN` instead of stopping the whole run.

In [15]:
sem = asyncio.Semaphore(6)

async def score_row(r):
    async with sem:
        async def safe(coro):
            try:
                return (await coro).value
            except Exception as e:
                return float("nan")
        f, a, p, c = await asyncio.gather(
            safe(faith.ascore(user_input=r.question, response=r.answer, retrieved_contexts=r.contexts)),
            safe(relev.ascore(user_input=r.question, response=r.answer)),
            safe(prec.ascore(user_input=r.question, reference=r.reference, retrieved_contexts=r.contexts)),
            safe(recall.ascore(user_input=r.question, reference=r.reference, retrieved_contexts=r.contexts)))
        return {"faithfulness": f, "answer_relevancy": a, "context_precision": p, "context_recall": c}

scores = await asyncio.gather(*[score_row(r) for r in runs.itertuples()])
results = pd.concat([runs[["config", "qid", "question", "answer"]], pd.DataFrame(scores)], axis=1)
results.to_csv("ragas_scores.csv", index=False)
print("scored rows:", len(results), "| NaN cells:", int(results.iloc[:, 4:].isna().sum().sum()))

scored rows: 48 | NaN cells: 0


### Average scores per configuration

In [16]:
METRICS = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
summary = results.groupby("config")[METRICS].mean().round(2).loc[list(configs)]
summary["context_chars"] = [round(runs[runs.config == c].contexts.apply(lambda cs: sum(map(len, cs))).mean()) for c in summary.index]
summary

,faithfulness,answer_relevancy,context_precision,context_recall,context_chars
config,,,,,
minilm_k4,0.93,0.58,0.37,0.50,2545
gemini_k2,0.97,0.74,0.71,0.75,1405
gemini_k4,1.00,0.66,0.69,0.67,2803
gemini_k8,1.00,0.80,0.59,0.67,5784


**What to notice:** three findings.

1. **The embedder is the biggest lever.** Same chunks, same prompt, same generator, same k=4: switching MiniLM to Gemini embeddings raises context precision from 0.37 to 0.69 and context recall from 0.50 to 0.67, and faithfulness from 0.93 to 1.00.
2. **The generator is not the problem.** Faithfulness is 0.93 or higher everywhere: Gemini writes answers from the context it is given. What goes wrong is what it is given.
3. **More chunks do not help after a point.** Going from k=2 to k=4 to k=8 keeps recall at 0.75, 0.67, 0.67 (differences of one question, within noise), lowers precision from 0.71 to 0.69 to 0.59, and doubles the prompt size each time (1,405, 2,803, 5,784 characters). `gemini_k2` is the best value here, `gemini_k4` the safer choice if questions need several passages.

No configuration passes 0.75 recall, so there is a ceiling that a bigger k does not fix. Section 8 looks at why. Remember the noise level: with 12 questions, one question changes a mean by 0.08, and `answer_relevancy` in particular moves around (0.74, 0.66, 0.80) without a trend.

## 7. Reading the scores per question

The averages hide the story; the per-question table shows where each configuration succeeds. Retrieval metrics are shown for the two embedders at k=4.

In [17]:
pivot = results[results.config.isin(["minilm_k4", "gemini_k4"])].pivot(index="qid", columns="config", values=["context_recall", "faithfulness", "answer_relevancy"]).round(2)
pivot.insert(0, "question", [qa[i]["question"][:48] for i in pivot.index])
pivot

question context_recall           faithfulness           answer_relevancy          
config                                                        gemini_k4 minilm_k4    gemini_k4 minilm_k4        gemini_k4 minilm_k4
qid                                                                                                                                
0       What are the two RAG formulations compared in th            1.0       0.0          1.0      1.00             0.93      0.90
1       Which retriever does the RAG model use, and what            0.0       0.0          1.0      0.80             0.93      0.00
2       Which generator model does RAG use and how many             0.0       0.0          1.0      1.00             0.86      0.83
3       How was the Wikipedia knowledge source prepared             0.0       0.0          1.0      0.50             0.33      0.00
4       What does the index hot-swapping experiment show            1.0       1.0          1.0      1.00             0.90      0.70
5       What search problem is solved to find the top-K             0.0       1.0          1.0      1.00             0.00      0.73
6       By how much does LoRA reduce trainable parameter            1.0       0.0          1.0      0.83             0.68      0.77
7       Which weight matrices does LoRA adapt in most of            1.0       1.0          1.0      1.00             0.62      0.93
8       How are the LoRA matrices A and B initialised, a            1.0       0.0          1.0      1.00             0.83      0.00
9                Why does LoRA add no inference latency?            1.0       1.0          1.0      1.00             0.92      0.93
10      How small can the LoRA rank be according to the             1.0       1.0          1.0      1.00             0.31      0.51
11      How much did LoRA shrink the GPT-3 175B checkpoi            1.0       1.0          1.0      1.00             0.62      0.70

**What to notice:** read down the recall columns. Gemini fixes three questions that MiniLM misses (0, 6 and 8) and loses one (5, the MIPS question), so no embedder wins everywhere. Questions 1, 2 and 3 have recall 0 for both. Faithfulness is 1.0 in every Gemini cell, while MiniLM dips on questions 1, 3 and 6. Answer relevancy has zeros for hedged or refused answers (MiniLM on questions 1, 3 and 8; Gemini on question 5, where it says the context "does not explicitly state" the answer).

## 8. Diagnosing a bad score: retrieval or generation?

A decision rule you can use on any RAG system:

| Context recall | Faithfulness | Likely problem | Where to fix |
|---|---|---|---|
| low | any | The evidence was not retrieved | chunking, embedding model, `k`, filters |
| high | low | The model added things not in the context | prompt, temperature, model choice |
| high | high | Fine for this question | - |
| any | high, relevancy low | Answer is grounded but off-question | prompt, question rewriting |

Let us apply it to the six worst rows for `minilm_k4`, sorted by recall and then faithfulness.

In [18]:
worst = results[results.config == "minilm_k4"].sort_values(["context_recall", "faithfulness"]).head(6)
worst[["qid", "context_recall", "faithfulness", "answer_relevancy", "answer"]].assign(answer=lambda d: d.answer.str[:80])

,qid,context_recall,faithfulness,answer_relevancy,answer
3,3,0.0,0.500000,0.000000,I don't know based on the provided documents.
1,1,0.0,0.800000,0.000000,The RAG model uses a retriever pη(z|x) with parameters η that returns distri...
6,6,0.0,0.833333,0.773480,"LoRA reduces the number of trainable parameters by approximately 10,000 time..."
0,0,0.0,1.000000,0.898375,The two RAG formulations compared are RAG-Token and RAG-Sequence. RAG-Sequen...
2,2,0.0,1.000000,0.832003,"The RAG models use BART-large as the generator, which has 406M parameters. T..."
8,8,0.0,1.000000,0.000000,I don't know based on the provided documents.


**What to notice:** apply the decision table row by row.

- **Question 0** (two formulations): recall 0, faithfulness 1.0, relevancy 0.90. The model was faithful to its context and answered on topic, but the context was the wrong one, so the answer is wrong. Neither generation metric can detect this; only recall, which uses the reference, does. This is the most important pattern in RAG evaluation.
- **Questions 3 and 8**: refusals. Relevancy 0 and recall 0 together say the evidence was not retrieved, and the refusal was the right behaviour. Fix: retrieval.
- **Question 6**: the answer says memory fell "from 35TB to 354GB". That mixes up the paper's footnote about *storing 100 adapted models* with training memory. Faithfulness (0.83) caught part of it.
- **Question 1**: the answer says the documents "do not specify the architecture" of the retriever. Relevancy 0 and recall 0 again point at retrieval (the bi-encoder passage was not fetched).
- **Question 2**: the answer is right (BART-large, 406M parameters) and faithful, yet recall is 0, because the reference contains details the retrieved text does not (the reference says 400M, the appendix says 406M). Low recall does not always mean a wrong answer, so read the answer before deciding.

Let us look inside question 0, the row with perfect faithfulness and zero recall.

In [19]:
r = runs[(runs.config == "minilm_k4") & (runs.qid == 0)].iloc[0]          # question 0: faithful but wrong
print("Q:", r.question, "\nA:", r.answer[:300], "\nREF:", r.reference)
print("retrieved pages:", [c[:40].replace("\n", " ") for c in r.contexts])

Q: What are the two RAG formulations compared in the RAG paper, and how do they differ? 
A: The two RAG formulations compared are RAG-Token and RAG-Sequence. RAG-Sequence is an improved version that is easier to use and has been open-sourced. 
REF: RAG-Sequence uses the same retrieved document to generate the whole output sequence, while RAG-Token can use a different retrieved document for each target token.
retrieved pages: ['Broader Impact This work offers several ', 'Table 4: Human assessments for the Jeopa', 'and easier to use implementation. This v', 'and TriviaQA. RAG compares favourably to']


**What to notice:** the question asks how the two formulations differ, yet the retrieved pages are the broader-impact section, a table of human assessments, an appendix paragraph about an implementation, and a results paragraph. None describes RAG-Sequence versus RAG-Token. The real description sits in Section 2.1, whose chunk did not rank in the top four for MiniLM. Retrieval failure is a **data and index** problem: no prompt change will fix it. Candidates are a better embedder (as we saw), smaller or smarter chunks, query rewriting and hybrid keyword plus vector search.

## 9. Limits of LLM-as-judge

RAGAs scores are produced by an LLM, so they inherit LLM weaknesses.

- **Judge noise.** The same input can score slightly differently on another run or with another judge model. We cache the judge so our numbers are reproducible, but treat differences of a few hundredths as noise.
- **Judge bias.** A judge from the same model family as the generator may be lenient on its own style. Use a different, stronger model as the judge when it matters.
- **Small samples.** With 12 questions, one question moves a mean by 0.08. Use RAGAs to find *which kind* of failure dominates, then confirm on more questions.
- **Refusals.** An honest "I don't know" scores 0 on answer relevancy by design, even though refusing is the correct behaviour when evidence is missing. Interpret those rows with the recall column beside them.
- **Wording sensitivity.** See the next cell: the same fact, worded differently, is scored very differently.
- **Cost.** One answer costs 8 to 14 judge calls here, more than the answer itself. Evaluate on a sample, not on every user query.

In [20]:
# the same fact as in section 1, but phrased with the word "only"
good_only = "In most experiments LoRA is applied only to the query and value projection matrices, Wq and Wv."
detail = await faith_detail(good_only)
print("faithfulness =", round(detail.verdict.mean(), 2))
detail

faithfulness = 0.0


,statement,verdict,reason
0,"In most experiments, LoRA is applied only to the query projection matrix, Wq.",0,"The context states ""we only apply LoRA toWq andWv in most experiments for si..."
1,"In most experiments, LoRA is applied only to the value projection matrix, Wv.",0,"The context states ""we only apply LoRA toWq andWv in most experiments for si..."


**What to notice:** the same fact, worded with the word "only", scores **0.0** against 1.0 for the wording in section 1. The judge split the sentence into "LoRA is applied only to Wq" and "LoRA is applied only to Wv", each of which is false on its own, because LoRA is applied to both. The error is in the step that decomposes the answer, not in the retrieval or the generation. This is a real limitation of statement-level faithfulness: a perfectly faithful answer can be penalised by an awkward split, so when a score looks wrong, look at the statements and the `reason` column before blaming your pipeline.

## Summary

| Idea | What we saw |
|---|---|
| Each metric has a mechanism | Faithfulness = supported statements / statements; relevancy = similarity of regenerated questions; precision = rank-aware average precision; recall = reference sentences attributable to context |
| Perturbations validate the metric | Adding a made-up claim, padding, reversing chunk order and swapping in wrong chunks each moved exactly the expected metric |
| Retrieval drives quality | Changing the embedder moved recall and precision far more than changing the generator prompt could |
| Scores guide decisions | We compared four configurations with the same question set and chose by evidence |

**Experiment 4 complete.** You built a PDF knowledge base (Part 1), a LangChain RAG chain with Gemini (Part 2) and an evaluation with RAGAs (Part 3).